# JSON / NDJSON Export Experiment

This notebook checks how Pandas exports mixed Python, NumPy, and Pandas objects to JSON Lines.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

out = Path("jsonExperiment_outputs")
out.mkdir(exist_ok=True)

In [2]:
df = pd.DataFrame([
    {
        "number": 3.14,
        "numpy_array_1": np.array([1.23]),
        "numpy_float": np.float64(4.56),
        "timestamp": pd.Timestamp("2026-05-13 12:34:56.789"),
        "string": "hello",
        "list_1": [7.89],
        "dict": {
            "number": 1,
            "numpy_array_1": np.array([2.0]),
            "numpy_float": np.float64(3.0),
            "timestamp": pd.Timestamp("2026-05-13 01:02:03"),
            "string": "inside dict",
            "list_1": [4.0],
        },
        "list": [
            1,
            np.array([2.0]),
            np.float64(3.0),
            pd.Timestamp("2026-05-13 04:05:06"),
            "inside list",
            [4.0],
            {"nested": np.array([5.0])},
        ],
    },
    {
        "number": 2.72,
        "numpy_array_1": np.array([9.87]),
        "numpy_float": np.float64(6.54),
        "timestamp": pd.Timestamp("2026-05-13 13:34:56.789"),
        "string": "world",
        "list_1": [1.11],
        "dict": {"number": 2, "numpy_array_1": np.array([8.0]), "timestamp": pd.Timestamp("2026-05-13 02:03:04")},
        "list": [2, np.array([8.0]), np.float64(9.0), pd.Timestamp("2026-05-13 05:06:07")],
    },
])

df

,number,numpy_array_1,numpy_float,timestamp,string,list_1,dict,list
0,3.14,[1.23],4.56,2026-05-13 12:34:56.789,hello,[7.89],"{'number': 1, 'numpy_array_1': [2.0], 'numpy_f...","[1, [2.0], 3.0, 2026-05-13 04:05:06, inside li..."
1,2.72,[9.87],6.54,2026-05-13 13:34:56.789,world,[1.11],"{'number': 2, 'numpy_array_1': [8.0], 'timesta...","[2, [8.0], 9.0, 2026-05-13 05:06:07]"


In [3]:
def try_export(name, fn):
    path = out / name
    try:
        fn(path)
        return {"name": name, "ok": True, "text": path.read_text()[:1000]}
    except Exception as e:
        return {"name": name, "ok": False, "error": type(e).__name__, "message": str(e)}


results = []

results.append(try_export(
    "records_lines_default.ndjson",
    lambda p: df.to_json(p, orient="records", lines=True),
))

results.append(try_export(
    "records_lines_iso.ndjson",
    lambda p: df.to_json(p, orient="records", lines=True, date_format="iso"),
))

results.append(try_export(
    "records_lines_epoch_ms.ndjson",
    lambda p: df.to_json(p, orient="records", lines=True, date_format="epoch", date_unit="ms"),
))

results.append(try_export(
    "records_json_array.json",
    lambda p: df.to_json(p, orient="records", date_format="iso"),
))

results.append(try_export(
    "table.json",
    lambda p: df.to_json(p, orient="table"),
))

pd.DataFrame(results)

C:\Users\RALPHG~1\AppData\Local\Temp\ipykernel_27812\3294047016.py:14: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  lambda p: df.to_json(p, orient="records", lines=True),
C:\Users\RALPHG~1\AppData\Local\Temp\ipykernel_27812\3294047016.py:24: Pandas4Warning: 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  lambda p: df.to_json(p, orient="records", lines=True, date_format="epoch", date_unit="ms"),


,name,ok,text
0,records_lines_default.ndjson,True,"{""number"":3.14,""numpy_array_1"":[1.23],""numpy_f..."
1,records_lines_iso.ndjson,True,"{""number"":3.14,""numpy_array_1"":[1.23],""numpy_f..."
2,records_lines_epoch_ms.ndjson,True,"{""number"":3.14,""numpy_array_1"":[1.23],""numpy_f..."
3,records_json_array.json,True,"[{""number"":3.14,""numpy_array_1"":[1.23],""numpy_..."
4,table.json,True,"{""schema"":{""fields"":[{""name"":""index"",""type"":""i..."


In [4]:
for r in results:
    print("\n" + "=" * 80)
    print(r["name"], "OK" if r["ok"] else "FAILED")
    print(r.get("text", r.get("message")))


records_lines_default.ndjson OK
{"number":3.14,"numpy_array_1":[1.23],"numpy_float":4.56,"timestamp":1778675696789,"string":"hello","list_1":[7.89],"dict":{"number":1,"numpy_array_1":[2.0],"numpy_float":3.0,"timestamp":1778634123000,"string":"inside dict","list_1":[4.0]},"list":[1,[2.0],3.0,1778645106000,"inside list",[4.0],{"nested":[5.0]}]}
{"number":2.72,"numpy_array_1":[9.87],"numpy_float":6.54,"timestamp":1778679296789,"string":"world","list_1":[1.11],"dict":{"number":2,"numpy_array_1":[8.0],"timestamp":1778637784000},"list":[2,[8.0],9.0,1778648767000]}


records_lines_iso.ndjson OK
{"number":3.14,"numpy_array_1":[1.23],"numpy_float":4.56,"timestamp":"2026-05-13T12:34:56.789","string":"hello","list_1":[7.89],"dict":{"number":1,"numpy_array_1":[2.0],"numpy_float":3.0,"timestamp":"2026-05-13T01:02:03.000","string":"inside dict","list_1":[4.0]},"list":[1,[2.0],3.0,"2026-05-13T04:05:06.000","inside list",[4.0],{"nested":[5.0]}]}
{"number":2.72,"numpy_array_1":[9.87],"numpy_float":6.5

In [5]:
round_trips = []
for r in results:
    if not r["ok"] or not r["name"].endswith(".ndjson"):
        continue
    path = out / r["name"]
    try:
        read = pd.read_json(path, lines=True)
        flat = pd.json_normalize(read.to_dict("records"))
        round_trips.append({"name": r["name"], "columns": list(flat.columns), "head": flat.head()})
    except Exception as e:
        round_trips.append({"name": r["name"], "error": repr(e)})

round_trips

[{'name': 'records_lines_default.ndjson',
  'columns': ['number',
   'numpy_array_1',
   'numpy_float',
   'timestamp',
   'string',
   'list_1',
   'list',
   'dict.number',
   'dict.numpy_array_1',
   'dict.numpy_float',
   'dict.timestamp',
   'dict.string',
   'dict.list_1'],
  'head':    number numpy_array_1  numpy_float               timestamp string  list_1  \
  0    3.14        [1.23]         4.56 2026-05-13 12:34:56.789  hello  [7.89]   
  1    2.72        [9.87]         6.54 2026-05-13 13:34:56.789  world  [1.11]   
  
                                                  list  dict.number  \
  0  [1, [2.0], 3.0, 1778645106000, inside list, [4...            1   
  1                     [2, [8.0], 9.0, 1778648767000]            2   
  
    dict.numpy_array_1  dict.numpy_float  dict.timestamp  dict.string  \
  0              [2.0]               3.0   1778634123000  inside dict   
  1              [8.0]               NaN   1778637784000          NaN   
  
    dict.list_1  
  0      

In [6]:
flat = pd.json_normalize(pd.read_json(out / "records_lines_iso.ndjson", lines=True).to_dict("records"))
flat

,number,numpy_array_1,numpy_float,timestamp,string,list_1,list,dict.number,dict.numpy_array_1,dict.numpy_float,dict.timestamp,dict.string,dict.list_1
0,3.14,[1.23],4.56,2026-05-13 12:34:56.789,hello,[7.89],"[1, [2.0], 3.0, 2026-05-13T04:05:06.000, insid...",1,[2.0],3.0,2026-05-13T01:02:03.000,inside dict,[4.0]
1,2.72,[9.87],6.54,2026-05-13 13:34:56.789,world,[1.11],"[2, [8.0], 9.0, 2026-05-13T05:06:07.000]",2,[8.0],NaN,2026-05-13T02:03:04.000,NaN,NaN


In [2]:
import pandas as pd
df = pd.DataFrame({"a": [1, 2], "b": ["x", "y"]})
df

,a,b
0,1,x
1,2,y
